# Core 08 - Single Agentic System

Objetivo: ejecutar un System completo con un provider seleccionable y evaluar el mismo Agent mediante `toolkit.eval`.

**Lugar en el modelo:** un System de una sola unidad es el caso mínimo: contiene un Agent, pero conserva ownership y una frontera de composición distinta.

**Evidencia exigida:** el Agent y su evaluación deben usar el Provider seleccionado y producir resultados verificables.

**Límite de la evidencia:** que Agent y System coincidan en este caso degenerado no los convierte en el mismo concepto.

## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| AGENTIC_SYSTEMS_SINGLE_PROVIDER | python-runtime | Cambiar el provider del sistema sin reescribirlo. |
| symbol | agent | Entrada evaluada por una Tool real. |
| eval records | dos casos | Validar exito y rechazo contra oracle. |

## 1) Seleccionar provider

Por default usa `python-runtime`. Para una prueba live define `AGENTIC_SYSTEMS_SINGLE_PROVIDER` como `openai-runtime`, `ollama-runtime`, `bedrock-runtime` o `vllm-runtime` junto con su configuración.

In [ ]:

import os

import agentic_systems as toolkit

PROVIDER = os.getenv("AGENTIC_SYSTEMS_SINGLE_PROVIDER", "python-runtime")
runtime = toolkit.runtime(provider=PROVIDER)
system = toolkit.system(runtime=runtime)
toolkit.show_json(runtime.describe(), title="Single-system runtime")

## 2) Construir Tool y Agent

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}

agent = system.agent(
    name="single_system_inspector",
    instructions="Usa inspect_public_api y responde con la evidencia observada.",
    tools=[inspect_public_api],
    runtime=runtime,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
    policy=toolkit.RunPolicy(max_tool_calls=1, max_turns=3, temperature=0.0),
)

def request_for(symbol: str):
    if PROVIDER == "python-runtime":
        return {"tool": "inspect_public_api", "input": {"symbol": symbol}}
    return f"Usa inspect_public_api para verificar el simbolo {symbol}."

result = agent.run(request_for("system"), mode="eval")
assert result.ok, result.errors
assert result.engine == PROVIDER
toolkit.human_result(result, title="Single-system RunResult", show_lineage=True)

## 3) Evaluar el mismo Agent

El oracle se declara en los casos y no se entrega como input al Agent.

In [ ]:
eval_cases = [
    {
        "name": "system_is_public",
        "input": request_for("system"),
        "expected": {
            "must_call": ["inspect_public_api"],
            "data_contains": {"tool": "inspect_public_api", "symbol": "system", "is_public": True, "ok": True},
        },
    },
    {
        "name": "unknown_is_not_public",
        "input": request_for("unknown_public_symbol"),
        "expected": {
            "must_call": ["inspect_public_api"],
            "data_contains": {"tool": "inspect_public_api", "symbol": "unknown_public_symbol", "is_public": False, "ok": True},
        },
    },
]

report = toolkit.eval().run(agent, eval_cases)
toolkit.show_json(toolkit.eval_report_summary(report), title="Single-system eval")
report.raise_if_failed()

## 4) API realmente ejercitada

In [ ]:
api_coverage = [
    "toolkit.runtime", "toolkit.system", "toolkit.tool", "system.agent",
    "agent.run", "toolkit.human_result", "toolkit.eval().run",
    "toolkit.eval_report_summary",
    "toolkit.AgentContract",
    "toolkit.RunPolicy",
    "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="Single-system API coverage")

## Resultado e interpretacion

Un RunResult real y un eval 2/2. Al cambiar provider cambia el backend, no la gramatica publica.